<a href="https://colab.research.google.com/github/vrnc-juga/Modelos-CNN/blob/CNN-100x100/Modelo_clasificacion_CNN_100x100.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
from matplotlib import pyplot as plt
import cv2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout
import os
from google.colab import drive
from tensorflow.keras.utils import to_categorical
import imghdr
from tensorflow.keras.callbacks import TensorBoard
from tensorflow import keras
from tensorflow.keras import layers

In [ ]:
drive.mount('/content/drive')
b=np.load('/content/drive/MyDrive/imagenes_ent2.npy', allow_pickle=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
X = [] #imagenes de entrada (pixeles)
y = [] #etiquetas
for img, label in b:
  X.append(img)
  y.append([label])

In [ ]:
X = np.array(X).astype(float) / 255 #normalizar

In [ ]:
y = to_categorical(y, num_classes=7)

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

WIDTH = 100
HEIGHT = 100
CHANNELS = 1
NUM_CLASSES = 7

inputs = keras.Input(shape=(WIDTH, HEIGHT, CHANNELS))

x = layers.Conv2D(32, (3,3), activation='relu', padding='same')(inputs)
x = layers.MaxPooling2D((2,2))(x)

x = layers.Conv2D(64, (3,3), activation='relu', padding='same')(x)
x = layers.MaxPooling2D((2,2))(x)

x = layers.Conv2D(128, (3,3), activation='relu', padding='same')(x)
x = layers.MaxPooling2D((2,2))(x)

x = layers.Conv2D(128, (3,3), activation='relu', padding='same')(x)

x = layers.Flatten()(x)

x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.5)(x)

outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = keras.Model(inputs, outputs)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 100, 100, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 100, 100, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 50, 50, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 50, 50, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 25, 25, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 25, 25, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 12, 12, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 18432)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     2,359,424 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,600,583 (9.92 MB)

 Trainable params: 2,600,583 (9.92 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(optimizer='adam',
                    loss='categorical_crossentropy',
                    # loss='sparse_categorical_crossentropy',
                    metrics=['accuracy'])


In [ ]:
# Asignación de índices para el entrenamiento y la prueba basada en la proporción deseada
split_idx = int(len(X) * 0.90)  # 85% para entrenamiento y 15% para prueba
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

In [ ]:
#aleatorios
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)

In [ ]:
#Realizar el aumento de datos con varias transformaciones. Al final, graficar 10 como ejemplo
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range=15,  # Rotación máxima de 10 grados
    width_shift_range=0.3,  # Desplazamiento horizontal máximo del 30%
    zoom_range=0.5,  # Rango de zoom
    horizontal_flip=True,  # Permitir volteo horizontal
    fill_mode='nearest',  # Modo de llenado de píxeles faltantes
    # rescale=1./255,
    height_shift_range=0.2,
    )
datagen.fit(X_train)

In [ ]:
class SaveBestModel(tf.keras.callbacks.Callback):
    def __init__(self, save_best_metric='val_loss', this_max=False):
        self.save_best_metric = save_best_metric
        self.max = this_max
        if this_max:
            self.best = float('-inf')
        else:
            self.best = float('inf')

    def on_epoch_end(self, epoch, logs=None):
        if self.save_best_metric in logs:
            metric_value = logs[self.save_best_metric]
            if self.max:
                if metric_value > self.best:
                    self.best = metric_value
                    self.best_weights = self.model.get_weights()
            else:
                if metric_value < self.best:
                    self.best = metric_value
                    self.best_weights = self.model.get_weights()
        else:
            print(f"La métrica {self.save_best_metric} no está disponible en logs.")


In [ ]:
tensorboardAD = TensorBoard(log_dir='logs/cnn', write_graph=False)
save_best_model = SaveBestModel()

model.fit(datagen.flow(X_train, y_train, batch_size=32),
          validation_data=(X_test, y_test),
          epochs=200,
          callbacks=[tensorboardAD, save_best_model])

Epoch 1/200
180/180 ━━━━━━━━━━━━━━━━━━━━ 8s 44ms/step - accuracy: 0.8492 - loss: 0.4458 - val_accuracy: 0.8928 - val_loss: 0.3743
Epoch 2/200
180/180 ━━━━━━━━━━━━━━━━━━━━ 8s 42ms/step - accuracy: 0.8424 - loss: 0.4757 - val_accuracy: 0.8788 - val_loss: 0.4096
Epoch 3/200
180/180 ━━━━━━━━━━━━━━━━━━━━ 9s 47ms/step - accuracy: 0.8474 - loss: 0.4544 - val_accuracy: 0.8928 - val_loss: 0.4287
Epoch 4/200
180/180 ━━━━━━━━━━━━━━━━━━━━ 7s 38ms/step - accuracy: 0.8408 - loss: 0.4653 - val_accuracy: 0.9123 - val_loss: 0.2646
Epoch 5/200
180/180 ━━━━━━━━━━━━━━━━━━━━ 8s 47ms/step - accuracy: 0.8411 - loss: 0.4619 - val_accuracy: 0.8851 - val_loss: 0.3670
Epoch 6/200
180/180 ━━━━━━━━━━━━━━━━━━━━ 7s 38ms/step - accuracy: 0.8586 - loss: 0.4333 - val_accuracy: 0.9032 - val_loss: 0.3270
Epoch 7/200
180/180 ━━━━━━━━━━━━━━━━━━━━ 8s 47ms/step - accuracy: 0.8464 - loss: 0.4482 - val_accuracy: 0.8886 - val_loss: 0.3353
Epoch 8/200
180/180 ━━━━━━━━━━━━━━━━━━━━ 7s 37ms/step - accuracy: 0.8492 - loss: 0.4359 - 

In [ ]:
model.set_weights(save_best_model.best_weights)
model.save_weights('best_weights.weights.h5')

In [ ]:
loss, acc = model.evaluate(X_test, y_test, verbose=2)
print("Restored model, accuracy: {:5.2f}%".format(100 * acc))


45/45 - 0s - 6ms/step - accuracy: 0.9227 - loss: 0.2560
Restored model, accuracy: 92.27%


In [ ]:
model.save('CLSF_GATOS.h5')